**Preprocessing**

In [82]:
!pip install datasets

In [83]:
from datasets import load_dataset, load_dataset_builder, get_dataset_config_names, get_dataset_split_names
import pandas as pd

In [84]:
import string
import nltk
import spacy

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

Get Dataset

In [85]:
ds_builder = load_dataset_builder("squad_v2")

In [86]:
ds_builder.info.features

{'id': Value(dtype='string', id=None),
 'title': Value(dtype='string', id=None),
 'context': Value(dtype='string', id=None),
 'question': Value(dtype='string', id=None),
 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}

In [87]:
get_dataset_split_names("squad_v2")

['train', 'validation']

In [88]:
train_data = load_dataset("squad_v2", split="train")

In [89]:
train = pd.DataFrame(train_data)
train

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,"{'text': ['in the late 1990s'], 'answer_start'..."
1,56be85543aeaaa14008c9065,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,"{'text': ['singing and dancing'], 'answer_star..."
2,56be85543aeaaa14008c9066,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...,"{'text': ['2003'], 'answer_start': [526]}"
3,56bf6b0f3aeaaa14008c9601,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?,"{'text': ['Houston, Texas'], 'answer_start': [..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?,"{'text': ['late 1990s'], 'answer_start': [276]}"
...,...,...,...,...,...
130314,5a7e070b70df9f001a875439,Matter,"The term ""matter"" is used throughout physics i...",Physics has broadly agreed on the definition o...,"{'text': [], 'answer_start': []}"
130315,5a7e070b70df9f001a87543a,Matter,"The term ""matter"" is used throughout physics i...",Who coined the term partonic matter?,"{'text': [], 'answer_start': []}"
130316,5a7e070b70df9f001a87543b,Matter,"The term ""matter"" is used throughout physics i...",What is another name for anti-matter?,"{'text': [], 'answer_start': []}"
130317,5a7e070b70df9f001a87543c,Matter,"The term ""matter"" is used throughout physics i...",Matter usually does not need to be used in con...,"{'text': [], 'answer_start': []}"


In [90]:
train['context'][200]

'At the 52nd Annual Grammy Awards, Beyoncé received ten nominations, including Album of the Year for I Am... Sasha Fierce, Record of the Year for "Halo", and Song of the Year for "Single Ladies (Put a Ring on It)", among others. She tied with Lauryn Hill for most Grammy nominations in a single year by a female artist. In 2010, Beyoncé was featured on Lady Gaga\'s single "Telephone" and its music video. The song topped the US Pop Songs chart, becoming the sixth number-one for both Beyoncé and Gaga, tying them with Mariah Carey for most number-ones since the Nielsen Top 40 airplay chart launched in 1992. "Telephone" received a Grammy Award nomination for Best Pop Collaboration with Vocals.'

Extract only necessary columns

In [91]:
prep = train[['context', 'question']]
prep

,context,question
0,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?
1,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...
2,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...
3,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?
4,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?
...,...,...
130314,"The term ""matter"" is used throughout physics i...",Physics has broadly agreed on the definition o...
130315,"The term ""matter"" is used throughout physics i...",Who coined the term partonic matter?
130316,"The term ""matter"" is used throughout physics i...",What is another name for anti-matter?
130317,"The term ""matter"" is used throughout physics i...",Matter usually does not need to be used in con...


Convert to lower case

In [92]:
prep = prep.applymap(lambda x: x.lower())
prep

,context,question
0,beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ b...,when did beyonce start becoming popular?
1,beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ b...,what areas did beyonce compete in when she was...
2,beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ b...,when did beyonce leave destiny's child and bec...
3,beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ b...,in what city and state did beyonce grow up?
4,beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ b...,in which decade did beyonce become famous?
...,...,...
130314,"the term ""matter"" is used throughout physics i...",physics has broadly agreed on the definition o...
130315,"the term ""matter"" is used throughout physics i...",who coined the term partonic matter?
130316,"the term ""matter"" is used throughout physics i...",what is another name for anti-matter?
130317,"the term ""matter"" is used throughout physics i...",matter usually does not need to be used in con...


Segmentation

In [93]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [95]:
prep = prep.applymap(lambda x: nltk.sent_tokenize(x) if isinstance(x, str) else x)
prep['context'][200]

['at the 52nd annual grammy awards, beyoncé received ten nominations, including album of the year for i am... sasha fierce, record of the year for "halo", and song of the year for "single ladies (put a ring on it)", among others.',
 'she tied with lauryn hill for most grammy nominations in a single year by a female artist.',
 'in 2010, beyoncé was featured on lady gaga\'s single "telephone" and its music video.',
 'the song topped the us pop songs chart, becoming the sixth number-one for both beyoncé and gaga, tying them with mariah carey for most number-ones since the nielsen top 40 airplay chart launched in 1992.',
 '"telephone" received a grammy award nomination for best pop collaboration with vocals.']

Remove punctuation and digits

In [96]:
remove_digits = str.maketrans('', '',string.digits)
remove_punctuation = str.maketrans('', '', string.punctuation)

def rem_punct(l):
  for i in range(len(l)):
    l[i] = l[i].translate(remove_digits)
    l[i] = l[i].translate(remove_punctuation)
  return l

In [97]:
prep = prep.applymap(lambda x: rem_punct(x))
prep['context'][200]

['at the nd annual grammy awards beyoncé received ten nominations including album of the year for i am sasha fierce record of the year for halo and song of the year for single ladies put a ring on it among others',
 'she tied with lauryn hill for most grammy nominations in a single year by a female artist',
 'in  beyoncé was featured on lady gagas single telephone and its music video',
 'the song topped the us pop songs chart becoming the sixth numberone for both beyoncé and gaga tying them with mariah carey for most numberones since the nielsen top  airplay chart launched in ',
 'telephone received a grammy award nomination for best pop collaboration with vocals']

Tokenization

In [99]:
def tokenize(l):
  for i in range(len(l)):
    l[i] = word_tokenize(l[i])
  return l

In [100]:
prep = prep.applymap(lambda x: tokenize(x))
prep['context'][200]

[['at',
  'the',
  'nd',
  'annual',
  'grammy',
  'awards',
  'beyoncé',
  'received',
  'ten',
  'nominations',
  'including',
  'album',
  'of',
  'the',
  'year',
  'for',
  'i',
  'am',
  'sasha',
  'fierce',
  'record',
  'of',
  'the',
  'year',
  'for',
  'halo',
  'and',
  'song',
  'of',
  'the',
  'year',
  'for',
  'single',
  'ladies',
  'put',
  'a',
  'ring',
  'on',
  'it',
  'among',
  'others'],
 ['she',
  'tied',
  'with',
  'lauryn',
  'hill',
  'for',
  'most',
  'grammy',
  'nominations',
  'in',
  'a',
  'single',
  'year',
  'by',
  'a',
  'female',
  'artist'],
 ['in',
  'beyoncé',
  'was',
  'featured',
  'on',
  'lady',
  'gagas',
  'single',
  'telephone',
  'and',
  'its',
  'music',
  'video'],
 ['the',
  'song',
  'topped',
  'the',
  'us',
  'pop',
  'songs',
  'chart',
  'becoming',
  'the',
  'sixth',
  'numberone',
  'for',
  'both',
  'beyoncé',
  'and',
  'gaga',
  'tying',
  'them',
  'with',
  'mariah',
  'carey',
  'for',
  'most',
  'numberones'

Remove Stop Words

In [101]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [102]:
stop_words = set(stopwords.words('english'))
def rem_stop(l):
  x=[]
  for i in range(len(l)):
    x.append([word for word in l[i] if word not in stop_words])
  return x

In [103]:
prep = prep.applymap(lambda x: rem_stop(x))
prep['context'][200]

[['nd',
  'annual',
  'grammy',
  'awards',
  'beyoncé',
  'received',
  'ten',
  'nominations',
  'including',
  'album',
  'year',
  'sasha',
  'fierce',
  'record',
  'year',
  'halo',
  'song',
  'year',
  'single',
  'ladies',
  'put',
  'ring',
  'among',
  'others'],
 ['tied',
  'lauryn',
  'hill',
  'grammy',
  'nominations',
  'single',
  'year',
  'female',
  'artist'],
 ['beyoncé',
  'featured',
  'lady',
  'gagas',
  'single',
  'telephone',
  'music',
  'video'],
 ['song',
  'topped',
  'us',
  'pop',
  'songs',
  'chart',
  'becoming',
  'sixth',
  'numberone',
  'beyoncé',
  'gaga',
  'tying',
  'mariah',
  'carey',
  'numberones',
  'since',
  'nielsen',
  'top',
  'airplay',
  'chart',
  'launched'],
 ['telephone',
  'received',
  'grammy',
  'award',
  'nomination',
  'best',
  'pop',
  'collaboration',
  'vocals']]

Final Dataset after preprocessing

In [104]:
train['context'] = prep['context']
train['question'] = prep['question']
train

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,"[[beyoncé, giselle, knowlescarter, biːˈjɒnseɪ,...","[[beyonce, start, becoming, popular]]","{'text': ['in the late 1990s'], 'answer_start'..."
1,56be85543aeaaa14008c9065,Beyoncé,"[[beyoncé, giselle, knowlescarter, biːˈjɒnseɪ,...","[[areas, beyonce, compete, growing]]","{'text': ['singing and dancing'], 'answer_star..."
2,56be85543aeaaa14008c9066,Beyoncé,"[[beyoncé, giselle, knowlescarter, biːˈjɒnseɪ,...","[[beyonce, leave, destinys, child, become, sol...","{'text': ['2003'], 'answer_start': [526]}"
3,56bf6b0f3aeaaa14008c9601,Beyoncé,"[[beyoncé, giselle, knowlescarter, biːˈjɒnseɪ,...","[[city, state, beyonce, grow]]","{'text': ['Houston, Texas'], 'answer_start': [..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,"[[beyoncé, giselle, knowlescarter, biːˈjɒnseɪ,...","[[decade, beyonce, become, famous]]","{'text': ['late 1990s'], 'answer_start': [276]}"
...,...,...,...,...,...
130314,5a7e070b70df9f001a875439,Matter,"[[term, matter, used, throughout, physics, bew...","[[physics, broadly, agreed, definition]]","{'text': [], 'answer_start': []}"
130315,5a7e070b70df9f001a87543a,Matter,"[[term, matter, used, throughout, physics, bew...","[[coined, term, partonic, matter]]","{'text': [], 'answer_start': []}"
130316,5a7e070b70df9f001a87543b,Matter,"[[term, matter, used, throughout, physics, bew...","[[another, name, antimatter]]","{'text': [], 'answer_start': []}"
130317,5a7e070b70df9f001a87543c,Matter,"[[term, matter, used, throughout, physics, bew...","[[matter, usually, need, used, conjunction]]","{'text': [], 'answer_start': []}"
